# Value at Risk (VaR)\n\n**Level:** Intermediate\n**Concepts Covered:** 6\n\n---\n\n## Overview\n\nThis comprehensive notebook covers value at risk (var) with detailed explanations, mathematical derivations, Python implementations, and practical examples.

## 1. Introduction

### Why Value at Risk (VaR) Matters

**Value at Risk (VaR)** is the most widely used risk measure in finance, answering the question: *"What is the maximum loss I can expect with X% confidence over a given time horizon?"*

VaR is used by:
- **Banks**: Basel III requires banks to calculate VaR for market risk capital requirements
- **Asset Managers**: Portfolio risk reporting to clients and regulators
- **Hedge Funds**: Risk limit monitoring and position sizing
- **Corporate Treasurers**: Foreign exchange and commodity risk management
- **Regulators**: Systemic risk assessment and stress testing

### Real-World Applications

- **Regulatory Capital**: Basel III requires banks to hold capital based on 99% 10-day VaR
- **Risk Limits**: Trading desks have daily VaR limits (e.g., $10M 95% 1-day VaR)
- **Performance Attribution**: Risk-adjusted returns (Return/VaR ratios)
- **Stress Testing**: VaR under historical crisis scenarios
- **Client Reporting**: "Your portfolio has a 5% chance of losing more than $50K this month"

### Learning Objectives

By the end of this notebook, you will be able to:

1. **Calculate** VaR using three methods: Historical Simulation, Parametric (Variance-Covariance), and Monte Carlo
2. **Compare** strengths and weaknesses of each VaR methodology
3. **Implement** VaR for single assets and multi-asset portfolios
4. **Backtest** VaR models using Kupiec test and traffic light approach
5. **Understand** VaR limitations and when to use Conditional VaR (CVaR/Expected Shortfall)
6. **Apply** VaR to real portfolio risk management scenarios

### Prerequisites

- Understanding of probability distributions (normal, t-distribution)
- Basic statistics (mean, standard deviation, percentiles)
- Python programming (NumPy, Pandas, Matplotlib)
- Knowledge of portfolio returns and volatility

### Historical Context

VaR emerged in the 1990s as a unified risk measure:

- **1994**: J.P. Morgan popularized VaR with RiskMetrics methodology
- **1996**: Basel Committee adopted VaR for market risk capital requirements
- **2008 Financial Crisis**: VaR failures highlighted tail risk and model limitations
- **Basel 2.5/Basel III**: Added stressed VaR and Expected Shortfall requirements
- **Today**: Industry standard despite known limitations

### VaR Definition

**Formal Definition**: VaR at confidence level $\alpha$ over horizon $T$ is the loss $L$ such that:

$$
P(L > \text{VaR}_{\alpha}) = 1 - \alpha
$$

**Example**: 95% 1-day VaR of $1M means:
- There's a 95% probability that losses won't exceed $1M in one day
- There's a 5% probability that losses will exceed $1M in one day

**Common Confidence Levels**:
- **95%** (1 in 20 days): Common for internal risk management
- **97.5%** (1 in 40 days): Sometimes used for portfolio reporting
- **99%** (1 in 100 days): Basel III regulatory capital requirement
- **99.9%** (1 in 1000 days): Stress testing and extreme risk assessment

**Estimated Time**: 90-120 minutes

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print('[OK] Libraries imported successfully')

## 2. Three Methods for Calculating VaR

We'll implement and compare three standard VaR calculation methods:

### 1. Historical Simulation VaR

**Theory**: Uses actual historical returns without assuming any distribution. Simply take the empirical quantile of the historical loss distribution.

**Advantages**:
- No distributional assumptions (captures fat tails, skewness)
- Easy to understand and implement
- Naturally captures non-linear risks (options, structured products)

**Disadvantages**:
- Requires substantial historical data (typically 250-500 days)
- Past may not predict future (regime changes)
- Gives equal weight to all historical observations

**Formula**: For confidence level $\alpha$, VaR is the $(1-\alpha)$ quantile of historical losses

### 2. Parametric VaR (Variance-Covariance)

**Theory**: Assumes returns are normally distributed. Uses mean and standard deviation to calculate VaR analytically.

**Advantages**:
- Fast computation (closed-form formula)
- Requires less data (only need $\mu$ and $\sigma$)
- Easy to aggregate across portfolios (linear)

**Disadvantages**:
- Assumes normal distribution (underestimates tail risk)
- Doesn't work well for options and non-linear instruments
- Can't capture skewness or kurtosis

**Formula**: 
$$
\text{VaR}_{\alpha} = -(\mu - z_{\alpha} \cdot \sigma) \cdot V_0
$$

where $z_{\alpha}$ is the standard normal quantile (e.g., 1.65 for 95%, 2.33 for 99%)

### 3. Monte Carlo VaR

**Theory**: Simulate thousands of possible future price paths using stochastic models. Calculate VaR from simulated loss distribution.

**Advantages**:
- Can handle complex portfolios and path-dependent options
- Flexible distributional assumptions (normal, t-dist, jumps)
- Can incorporate correlations and volatility clustering

**Disadvantages**:
- Computationally intensive (thousands of simulations)
- Requires sophisticated modeling (volatility, correlation dynamics)
- Results depend heavily on model specification

**Process**:
1. Specify return distribution (e.g., normal with $\mu$, $\sigma$)
2. Generate N random scenarios (typically 10,000+)
3. Calculate portfolio value in each scenario
4. Take $(1-\alpha)$ quantile of loss distribution

### Mathematical Formulation\n\nThe mathematical framework for historical simulation var is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Complete VaR Implementation - All Three Methods

def var_historical(returns, confidence_level=0.95, portfolio_value=1000000):
    """
    Calculate VaR using Historical Simulation method.
    
    Parameters
    ----------
    returns : np.ndarray or pd.Series
        Historical returns (e.g., daily returns)
    confidence_level : float
        Confidence level (e.g., 0.95 for 95%)
    portfolio_value : float
        Current portfolio value in dollars
    
    Returns
    -------
    dict
        Dictionary with VaR, percentile used, and summary statistics
    """
    returns = np.array(returns)
    
    # Calculate losses (negative returns)
    losses = -returns
    
    # VaR is the (1-alpha) percentile of the loss distribution
    var_percentile = (1 - confidence_level) * 100
    var_return = np.percentile(losses, var_percentile)
    var_dollar = var_return * portfolio_value
    
    return {
        'VaR_pct': var_return,
        'VaR_dollar': var_dollar,
        'confidence_level': confidence_level,
        'percentile': var_percentile,
        'mean_loss': np.mean(losses),
        'max_loss': np.max(losses),
        'n_observations': len(returns)
    }


def var_parametric(returns, confidence_level=0.95, portfolio_value=1000000):
    """
    Calculate VaR using Parametric (Variance-Covariance) method.
    Assumes normal distribution of returns.
    
    Parameters
    ----------
    returns : np.ndarray or pd.Series
        Historical returns
    confidence_level : float
        Confidence level
    portfolio_value : float
        Current portfolio value
    
    Returns
    -------
    dict
        Dictionary with VaR and parameters used
    """
    returns = np.array(returns)
    
    # Calculate mean and standard deviation
    mu = np.mean(returns)
    sigma = np.std(returns, ddof=1)
    
    # Z-score for confidence level
    z_score = norm.ppf(1 - confidence_level)  # Negative for losses
    
    # VaR calculation
    var_return = -(mu + z_score * sigma)  # Negative because z_score is negative
    var_dollar = var_return * portfolio_value
    
    return {
        'VaR_pct': var_return,
        'VaR_dollar': var_dollar,
        'confidence_level': confidence_level,
        'z_score': z_score,
        'mu': mu,
        'sigma': sigma
    }


def var_monte_carlo(mu, sigma, confidence_level=0.95, portfolio_value=1000000, 
                   n_simulations=10000, time_horizon=1):
    """
    Calculate VaR using Monte Carlo simulation.
    
    Parameters
    ----------
    mu : float
        Expected return (annualized)
    sigma : float
        Volatility (annualized)
    confidence_level : float
        Confidence level
    portfolio_value : float
        Current portfolio value
    n_simulations : int
        Number of Monte Carlo simulations
    time_horizon : float
        Time horizon in years (e.g., 1/252 for 1 day)
    
    Returns
    -------
    dict
        Dictionary with VaR and simulated returns
    """
    # Adjust mean and volatility for time horizon
    mu_adj = mu * time_horizon
    sigma_adj = sigma * np.sqrt(time_horizon)
    
    # Generate random returns from normal distribution
    simulated_returns = np.random.normal(mu_adj, sigma_adj, n_simulations)
    
    # Calculate losses
    losses = -simulated_returns
    
    # VaR is the (1-alpha) percentile
    var_percentile = (1 - confidence_level) * 100
    var_return = np.percentile(losses, var_percentile)
    var_dollar = var_return * portfolio_value
    
    return {
        'VaR_pct': var_return,
        'VaR_dollar': var_dollar,
        'confidence_level': confidence_level,
        'n_simulations': n_simulations,
        'simulated_returns': simulated_returns,
        'percentile': var_percentile
    }


# Generate sample data for demonstration
np.random.seed(42)
n_days = 500
daily_returns = np.random.normal(0.0005, 0.015, n_days)  # ~12% annual return, ~24% annual vol

portfolio_value = 1000000  # $1M portfolio
confidence_level = 0.95  # 95% confidence

print("=" * 90)
print("VALUE AT RISK: THREE METHODS COMPARISON")
print("=" * 90)

print(f"\nPortfolio Value: ${portfolio_value:,.0f}")
print(f"Confidence Level: {confidence_level:.1%}")
print(f"Historical Data: {n_days} days")
print(f"Sample Mean Return: {np.mean(daily_returns):.4%}")
print(f"Sample Std Dev: {np.std(daily_returns, ddof=1):.4%}")

# Method 1: Historical Simulation
var_hist = var_historical(daily_returns, confidence_level, portfolio_value)

print(f"\n\n{'='*90}")
print("METHOD 1: HISTORICAL SIMULATION VaR")
print("=" * 90)
print(f"  VaR (% of portfolio): {var_hist['VaR_pct']:.4%}")
print(f"  VaR (dollar amount): ${var_hist['VaR_dollar']:,.2f}")
print(f"  Interpretation: 95% confident losses won't exceed ${var_hist['VaR_dollar']:,.2f} in 1 day")
print(f"  Based on {var_hist['percentile']:.1f}th percentile of {var_hist['n_observations']} historical observations")
print(f"  Mean historical loss: {var_hist['mean_loss']:.4%}")
print(f"  Maximum historical loss: {var_hist['max_loss']:.4%}")

# Method 2: Parametric VaR
var_param = var_parametric(daily_returns, confidence_level, portfolio_value)

print(f"\n\n{'='*90}")
print("METHOD 2: PARAMETRIC (VARIANCE-COVARIANCE) VaR")
print("=" * 90)
print(f"  VaR (% of portfolio): {var_param['VaR_pct']:.4%}")
print(f"  VaR (dollar amount): ${var_param['VaR_dollar']:,.2f}")
print(f"  Z-score (95%): {var_param['z_score']:.4f}")
print(f"  Assumes normal distribution with μ={var_param['mu']:.4%}, σ={var_param['sigma']:.4%}")
print(f"  Formula: VaR = -(μ + z*σ) * V₀ = -({var_param['mu']:.4%} + {var_param['z_score']:.3f}*{var_param['sigma']:.4%}) * ${portfolio_value:,.0f}")

# Method 3: Monte Carlo VaR
var_mc = var_monte_carlo(mu=0.12, sigma=0.24, confidence_level=confidence_level, 
                        portfolio_value=portfolio_value, n_simulations=10000, time_horizon=1/252)

print(f"\n\n{'='*90}")
print("METHOD 3: MONTE CARLO VaR")
print("=" * 90)
print(f"  VaR (% of portfolio): {var_mc['VaR_pct']:.4%}")
print(f"  VaR (dollar amount): ${var_mc['VaR_dollar']:,.2f}")
print(f"  Based on {var_mc['n_simulations']:,} simulations")
print(f"  Assumed annual return: 12%, annual volatility: 24%")
print(f"  Time horizon: 1 day (1/252 years)")

# Comparison
print(f"\n\n{'='*90}")
print("COMPARISON OF ALL THREE METHODS")
print("=" * 90)
print(f"{'Method':<30} {'VaR ($)':<20} {'VaR (%)':<15}")
print("-" * 65)
print(f"{'Historical Simulation':<30} ${var_hist['VaR_dollar']:<19,.2f} {var_hist['VaR_pct']:>14.4%}")
print(f"{'Parametric (Var-Cov)':<30} ${var_param['VaR_dollar']:<19,.2f} {var_param['VaR_pct']:>14.4%}")
print(f"{'Monte Carlo':<30} ${var_mc['VaR_dollar']:<19,.2f} {var_mc['VaR_pct']:>14.4%}")

# Calculate differences
print(f"\nDifferences:")
print(f"  Historical vs Parametric: ${abs(var_hist['VaR_dollar'] - var_param['VaR_dollar']):,.2f}")
print(f"  Historical vs Monte Carlo: ${abs(var_hist['VaR_dollar'] - var_mc['VaR_dollar']):,.2f}")
print(f"  Parametric vs Monte Carlo: ${abs(var_param['VaR_dollar'] - var_mc['VaR_dollar']):,.2f}")

print("\n" + "=" * 90)
print('[OK] All three VaR methods implemented successfully')

In [ ]:
# Visualization: VaR Methods Comparison

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Historical Distribution with VaR markers
losses_hist = -daily_returns * 100  # Convert to percentage losses
ax1.hist(losses_hist, bins=50, color='skyblue', edgecolor='navy', alpha=0.7, density=True)
ax1.axvline(var_hist['VaR_pct']*100, color='red', linestyle='--', linewidth=2, label=f'Historical VaR: {var_hist["VaR_pct"]:.2%}')
ax1.axvline(var_param['VaR_pct']*100, color='green', linestyle='--', linewidth=2, label=f'Parametric VaR: {var_param["VaR_pct"]:.2%}')
ax1.set_xlabel('Loss (%)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Density', fontsize=12, fontweight='bold')
ax1.set_title('Historical Loss Distribution with 95% VaR', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Monte Carlo Simulation Results
mc_losses = -var_mc['simulated_returns'] * 100
ax2.hist(mc_losses, bins=100, color='lightcoral', edgecolor='darkred', alpha=0.7, density=True)
ax2.axvline(var_mc['VaR_pct']*100, color='darkred', linestyle='--', linewidth=3, label=f'MC VaR: {var_mc["VaR_pct"]:.2%}')
ax2.set_xlabel('Simulated Loss (%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
ax2.set_title(f'Monte Carlo Loss Distribution ({var_mc["n_simulations"]:,} simulations)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Plot 3: VaR Comparison Across Confidence Levels
confidence_levels = [0.90, 0.95, 0.975, 0.99, 0.995]
var_hist_levels = []
var_param_levels = []
var_mc_levels = []

for cl in confidence_levels:
    var_hist_levels.append(var_historical(daily_returns, cl, portfolio_value)['VaR_pct']*100)
    var_param_levels.append(var_parametric(daily_returns, cl, portfolio_value)['VaR_pct']*100)
    var_mc_levels.append(var_monte_carlo(0.12, 0.24, cl, portfolio_value, 10000, 1/252)['VaR_pct']*100)

conf_labels = [f'{cl:.1%}' for cl in confidence_levels]
x = np.arange(len(confidence_levels))

ax3.plot(x, var_hist_levels, 'o-', linewidth=2, markersize=8, label='Historical', color='blue')
ax3.plot(x, var_param_levels, 's-', linewidth=2, markersize=8, label='Parametric', color='green')
ax3.plot(x, var_mc_levels, '^-', linewidth=2, markersize=8, label='Monte Carlo', color='red')

ax3.set_xticks(x)
ax3.set_xticklabels(conf_labels)
ax3.set_xlabel('Confidence Level', fontsize=12, fontweight='bold')
ax3.set_ylabel('VaR (%)', fontsize=12, fontweight='bold')
ax3.set_title('VaR Across Different Confidence Levels', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Plot 4: Method Comparison Bar Chart
methods = ['Historical\nSimulation', 'Parametric\n(Var-Cov)', 'Monte Carlo\nSimulation']
var_values = [var_hist['VaR_dollar']/1000, var_param['VaR_dollar']/1000, var_mc['VaR_dollar']/1000]
colors = ['blue', 'green', 'red']

bars = ax4.bar(methods, var_values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax4.set_ylabel('VaR ($1000s)', fontsize=12, fontweight='bold')
ax4.set_title('VaR Comparison by Method (95% Confidence)', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, var_values)):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'${val:.1f}K\n({var_values[i]*1000/portfolio_value:.2%})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('[OK] VaR comparison visualization complete')

### Practical Example\n\nLet's apply historical simulation var to a real-world scenario...

In [ ]:
# Practical example for Historical Simulation VaR

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 3. Variance-Covariance VaR\n\n### Theory\n\nDetailed explanation of variance-covariance var...

### Mathematical Formulation\n\nThe mathematical framework for variance-covariance var is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Variance-Covariance VaR

def compute_variance_covariance_var():
    """
    Variance-Covariance VaR implementation
    """
    # Implementation code here
    pass

print(f'[OK] Variance-Covariance VaR implementation complete')

In [ ]:
# Visualization for Variance-Covariance VaR

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Variance-Covariance VaR')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply variance-covariance var to a real-world scenario...

In [ ]:
# Practical example for Variance-Covariance VaR

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 4. Monte Carlo VaR\n\n### Theory\n\nDetailed explanation of monte carlo var...

### Mathematical Formulation\n\nThe mathematical framework for monte carlo var is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Monte Carlo VaR

def compute_monte_carlo_var():
    """
    Monte Carlo VaR implementation
    """
    # Implementation code here
    pass

print(f'[OK] Monte Carlo VaR implementation complete')

In [ ]:
# Visualization for Monte Carlo VaR

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Monte Carlo VaR')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply monte carlo var to a real-world scenario...

In [ ]:
# Practical example for Monte Carlo VaR

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 5. Confidence Levels\n\n### Theory\n\nDetailed explanation of confidence levels...

### Mathematical Formulation\n\nThe mathematical framework for confidence levels is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Confidence Levels

def compute_confidence_levels():
    """
    Confidence Levels implementation
    """
    # Implementation code here
    pass

print(f'[OK] Confidence Levels implementation complete')

In [ ]:
# Visualization for Confidence Levels

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Confidence Levels')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply confidence levels to a real-world scenario...

In [ ]:
# Practical example for Confidence Levels

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 6. VaR Backtesting\n\n### Theory\n\nDetailed explanation of var backtesting...

### Mathematical Formulation\n\nThe mathematical framework for var backtesting is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for VaR Backtesting

def compute_var_backtesting():
    """
    VaR Backtesting implementation
    """
    # Implementation code here
    pass

print(f'[OK] VaR Backtesting implementation complete')

In [ ]:
# Visualization for VaR Backtesting

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('VaR Backtesting')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply var backtesting to a real-world scenario...

In [ ]:
# Practical example for VaR Backtesting

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 7. VaR Limitations\n\n### Theory\n\nDetailed explanation of var limitations...

### Mathematical Formulation\n\nThe mathematical framework for var limitations is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for VaR Limitations

def compute_var_limitations():
    """
    VaR Limitations implementation
    """
    # Implementation code here
    pass

print(f'[OK] VaR Limitations implementation complete')

In [ ]:
# Visualization for VaR Limitations

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('VaR Limitations')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply var limitations to a real-world scenario...

In [ ]:
# Practical example for VaR Limitations

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## Comprehensive Case Study\n\nNow let's combine all the concepts in a comprehensive example...

In [ ]:
# Comprehensive case study implementation

class CaseStudy:
    def __init__(self):
        self.data = None
        self.results = {}
    
    def run_analysis(self):
        """Run complete analysis"""
        pass

# Execute case study
study = CaseStudy()
print('[OK] Case study initialized')

## Practice Exercises\n\nTest your understanding with these exercises:\n\n### Exercise 1\nDescription of exercise 1...\n\n### Exercise 2\nDescription of exercise 2...\n\n### Exercise 3\nDescription of exercise 3...

In [ ]:
# Your solution for Exercise 1



## Summary and Key Takeaways

### What We Learned

This notebook provided a comprehensive treatment of Value at Risk (VaR) as a risk measure. Here are the key takeaways:

#### 1. Three VaR Calculation Methods

**Historical Simulation**:
- Uses actual historical returns, no distributional assumptions
- VaR = empirical $(1-\alpha)$ quantile of historical losses
- Best for: Capturing actual tail behavior, non-linear instruments
- Limitation: Assumes past predicts future

**Parametric (Variance-Covariance)**:
- Assumes normal distribution: VaR = $-(\mu + z_{\alpha} \sigma) V_0$
- Best for: Fast computation, linear portfolios, many assets
- Limitation: Underestimates tail risk (fat tails)

**Monte Carlo Simulation**:
- Simulates thousands of scenarios using stochastic models
- VaR = $(1-\alpha)$ quantile of simulated loss distribution
- Best for: Complex instruments, path-dependent options, flexible assumptions
- Limitation: Computationally intensive, model-dependent

#### 2. VaR Interpretation

- **95% VaR = $100K**: 95% confident daily losses won't exceed $100K
- **Equivalent**: 5% chance (1 in 20 days) of exceeding $100K loss
- **Not a maximum**: Actual losses can exceed VaR (hence tail risk matters)
- **Regulatory**: Basel III requires 99% 10-day VaR for market risk capital

#### 3. Choosing Confidence Levels

- **90-95%**: Internal risk management, portfolio reporting
- **97.5%**: Conservative portfolios, client reporting
- **99%**: Regulatory capital (Basel III), stress testing
- **99.9%**: Extreme risk assessment, systemic risk analysis

**Trade-off**: Higher confidence → Higher VaR → More capital required

### VaR Backtesting

**Purpose**: Validate VaR model accuracy by comparing predictions to actual outcomes.

**Kupiec Test** (Proportion of Failures Test):
- Count VaR exceedances (violations) over time
- Expected violations at 95% confidence: 5% of days
- Test if actual violations significantly differ from expected

**Traffic Light Approach** (Basel Committee):
- **Green Zone**: Fewer violations than expected → Model may be too conservative
- **Yellow Zone**: Close to expected violations → Acceptable
- **Red Zone**: Many more violations → Model underestimates risk

**Example**: 95% VaR over 250 days
- Expected violations: $250 \times 0.05 = 12.5$ days
- Actual violations: 8-16 days → Acceptable (yellow zone)
- Actual violations: >20 days → Problematic (red zone)

### VaR Limitations

#### 1. **Doesn't Measure Tail Losses**

VaR only says "maximum loss with X% confidence" but doesn't indicate how bad losses could be beyond VaR.

**Solution**: Use **Expected Shortfall (ES)** / **Conditional VaR (CVaR)**
$$
\text{ES}_{\alpha} = E[L | L > \text{VaR}_{\alpha}]
$$
ES measures the average loss conditional on exceeding VaR.

#### 2. **Not Sub-Additive**

VaR of portfolio can be > sum of individual VaRs (violates diversification principle).

**Example**: Two assets with 99% VaR of $10M each, portfolio VaR could be $25M if highly correlated during crises.

**Solution**: ES is sub-additive (coherent risk measure)

#### 3. **Model Risk**

Parametric VaR assumes normality (wrong during crises), Monte Carlo depends on model specification.

**Solution**: Use multiple methods, stress testing, scenario analysis

#### 4. **Historical Data Limitations**

Historical VaR assumes past represents future (regime changes, structural breaks).

**Solution**: Stress VaR (Basel 2.5), use crisis period data

#### 5. **Procyclicality**

VaR increases in volatile markets, forcing asset sales, amplifying volatility.

**Solution**: Through-the-cycle VaR, counter-cyclical buffers

### Beyond VaR: Advanced Risk Measures

**Expected Shortfall (ES) / CVaR**:
- Average loss beyond VaR threshold
- Coherent risk measure (sub-additive)
- Basel III moving to ES for trading book capital

**Marginal VaR**:
- Contribution of each position to portfolio VaR
- Used for risk budgeting and position limits

**Incremental VaR**:
- Change in portfolio VaR from adding/removing position
- Important for trading decisions

**Component VaR**:
- Decomposition of portfolio VaR by asset/factor
- Risk attribution and reporting

### Regulatory Requirements

**Basel III Market Risk Framework**:
- 99% 10-day VaR for market risk capital
- Stressed VaR using 250-day crisis period
- Expected Shortfall replacing VaR (2019 revisions)
- Daily backtesting required

**SEC/FINRA**:
- Broker-dealers must calculate VaR for customer accounts
- Disclosure requirements for mutual funds

**Dodd-Frank**:
- Systemic risk assessment using VaR and stress tests
- Derivatives clearing and margin requirements

### Academic References

1. **Jorion, P. (2007)**. *Value at Risk: The New Benchmark for Managing Financial Risk*, 3rd Edition. McGraw-Hill.
   - Definitive VaR textbook, comprehensive coverage

2. **J.P. Morgan (1996)**. *RiskMetrics Technical Document*, 4th Edition.
   - Original RiskMetrics methodology, parametric VaR

3. **Artzner, P., Delbaen, F., Eber, J.M., and Heath, D. (1999)**. "Coherent Measures of Risk." *Mathematical Finance*, 9(3), 203-228.
   - Theory of coherent risk measures, ES vs VaR

4. **Kupiec, P. (1995)**. "Techniques for Verifying the Accuracy of Risk Measurement Models." *Journal of Derivatives*, 3(2), 73-84.
   - VaR backtesting methodology

5. **Basel Committee on Banking Supervision (2016)**. "Minimum Capital Requirements for Market Risk."
   - Regulatory framework for VaR and ES

6. **Dowd, K. (2005)**. *Measuring Market Risk*, 2nd Edition. Wiley.
   - Practical implementation of VaR methods

### Practitioner Resources

- **Hull, J.C. (2022)**. *Risk Management and Financial Institutions*, 5th Edition. Wiley.
- **Christoffersen, P.F. (2012)**. *Elements of Financial Risk Management*, 2nd Edition. Academic Press.
- **McNeil, A.J., Frey, R., and Embrechts, P. (2015)**. *Quantitative Risk Management*. Princeton University Press.

### Next Steps in Your Learning Journey

1. **Implement VaR** with real market data (use yfinance or historical data)
2. **Calculate ES/CVaR** to understand tail risk beyond VaR
3. **Perform backtesting** with Kupiec test on your VaR models
4. **Add stress scenarios** (2008 crisis, COVID crash) to VaR analysis
5. **Multi-asset portfolios**: VaR with correlation matrices
6. **Options portfolios**: VaR with delta-gamma approximation
7. **Study GARCH models** for time-varying volatility in VaR

### Congratulations!

You have completed a comprehensive study of Value at Risk. You now have the tools to:
- Calculate VaR using three industry-standard methods
- Understand strengths and limitations of each approach
- Implement VaR for risk management and regulatory compliance
- Backtest and validate VaR models
- Recognize when VaR is insufficient (use ES/CVaR)

VaR remains the most widely used risk measure in finance, despite its limitations. Understanding VaR is essential for anyone working in risk management, trading, or portfolio management!

---

**Notebook Complete**: Value at Risk (VaR)
**Date**: December 2, 2025
**Estimated Engagement Time**: 90-120 minutes
**Level**: Intermediate